# ORBAT Classification System - Training Pipeline

This notebook demonstrates the complete training pipeline for the ORBAT classification system.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import ORBATPreprocessor, prepare_train_test_split
from src.models import ClassificationModel, SimilarityModel
from src.hybrid_system import HybridORBATSystem
from src.evaluation import ORBATEvaluator
from src.inference import ORBATPredictor, build_hierarchy_map

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/orbat_sample.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of unique units: {df['unit_id'].nunique()}")
print("\nFirst few rows:")
df.head()

In [ ]:
# Data distribution
print("Unit distribution:")
print(df['unit_id'].value_counts().head(10))

print("\nEquipment types:")
print(df['equipment_type'].value_counts())

print("\nHierarchy levels:")
print(df['hierarchy_level'].value_counts())

## 2. Data Preprocessing

In [ ]:
# Split data
train_val_df, test_df = prepare_train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = prepare_train_test_split(train_val_df, test_size=0.125, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Preprocess
preprocessor = ORBATPreprocessor()
X_train, y_train = preprocessor.fit_transform(train_df)
X_val, y_val = preprocessor.transform(val_df), preprocessor.target_encoder.transform(val_df['unit_id'])
X_test, y_test = preprocessor.transform(test_df), preprocessor.target_encoder.transform(test_df['unit_id'])

print(f"Feature dimension: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y_train))}")

## 3. Train Classification Model

In [ ]:
# Train XGBoost classifier
clf_model = ClassificationModel(model_type='xgboost')
clf_model.train(X_train, y_train, X_val, y_val)

# Evaluate classification model alone
y_pred_clf = clf_model.predict(X_test)
accuracy_clf = (y_pred_clf == y_test).mean()
print(f"Classification Model Accuracy: {accuracy_clf:.4f}")

## 4. Train Similarity Model

In [ ]:
# Train similarity model
sim_model = SimilarityModel(metric='cosine', n_neighbors=5)
sim_model.fit(X_train, y_train)

print("Similarity model trained successfully")

## 5. Build Hybrid System

In [ ]:
# Create hybrid system
hybrid_system = HybridORBATSystem(clf_model, sim_model, alpha=0.6)

# Calibrate
calibration_metrics = hybrid_system.calibrate_confidence(X_val, y_val)
print("Calibration metrics:")
for key, value in calibration_metrics.items():
    print(f"  {key}: {value:.4f}")

## 6. Evaluate Hybrid System

In [ ]:
# Predict on test set
predictions, confidences, _ = hybrid_system.predict(X_test)
proba = clf_model.predict_proba(X_test)

# Evaluate
class_names = [str(c) for c in preprocessor.target_encoder.classes_]
evaluator = ORBATEvaluator(class_names=class_names)
metrics = evaluator.evaluate(y_test, predictions, proba, confidences)

evaluator.print_summary()

In [ ]:
# Plot confusion matrix
evaluator.plot_confusion_matrix(figsize=(10, 8))

In [ ]:
# Plot confidence distribution
evaluator.plot_confidence_distribution(confidences, y_test, predictions)

## 7. Test Inference

In [ ]:
# Build hierarchy map
hierarchy_map = build_hierarchy_map(df)

# Create predictor
predictor = ORBATPredictor(preprocessor, hybrid_system, hierarchy_map)

# Test prediction
test_observation = {
    'equipment_type': 'tank',
    'equipment_count': 12,
    'communication_frequency': 150.5,
    'mobility': 'mobile',
    'hierarchy_level': 'Battalion',
    'communication_degree': 6,
    'location_x': 450.2,
    'location_y': 320.8,
    'cluster_id': 2
}

result = predictor.predict(test_observation, return_details=True)
print("Prediction result:")
import json
print(json.dumps(result, indent=2))

## 8. Save Models

In [ ]:
# Save all models
from pathlib import Path

model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)

preprocessor.save(model_dir / 'preprocessor.pkl')
clf_model.save(model_dir / 'classification_model.pkl')
sim_model.save(model_dir / 'similarity_model.pkl')
predictor.save(model_dir / 'orbat_predictor.pkl')
evaluator.save_metrics(model_dir / 'evaluation_metrics.json')

print("All models saved successfully!")